> ### v2 update — one benchmark, all 185, YOLO threshold swept
> - **Benchmarks 2 & 3 removed.** One benchmark remains: raw forward over **all 185**
>   staged test tensors on the GPU (matches `benchmark_speed` in per_model_batch/final).
> - **YOLO threshold swept on val** (134 images, subject-disjoint) instead of bare argmax,
>   so YOLO is symmetric with SegFormer (both val-calibrated). argmax is just thr = 0.5.
>
> ## 🟢 VERSION 2 — YOLO inference made consistent with the dataloader
>
> Same notebook, same cells, same variables, same 3 benchmarks — only the **YOLO
> inference path** is changed to match the logic used in `bruise_colab_final.ipynb` /
> `bruise_colab_train_per_model_batch.ipynb`:
>
> - **No letterbox.** YOLO is preprocessed with the **same 640 stretch-resize** as the
>   dataloader/SegFormer (letterbox removed), so timing and scoring use one grid.
> - **Use the dataloader for testing too.** Accuracy no longer calls Ultralytics native
>   `.predict()`; it feeds the **dataloader tensor** (de-normalised back to `/255`, since
>   YOLO trained on `/255`) through the raw module, mask = `argmax` over the 2 channels.
>
> SegFormer is unchanged. **The original `bruise_colab_inference.ipynb` is untouched —
> keep it to compare** (YOLO Dice will differ: v1 = native letterbox `.predict()`,
> v2 = dataloader stretch `/255` argmax).

---

# Bruise Segmentation — Inference, Benchmarking & Evaluation Demo

This notebook runs our **5 trained models** and answers two questions:

1. **How fast are they?** (three different benchmarks — they measure different things)
2. **How accurate are they?** (Dice / IoU / miss-rate on the 185-image test set)

**Everything is written out in this notebook.** We deliberately do *not* call
`pipeline/benchmark_640.py` for the timing code — the timing loops are spelled out
in plain cells below so you can read exactly what is being measured. We only import
from `pipeline/` for the boring, well-tested parts (dataset loading, checkpoint
loading, metrics).

---

## The three benchmarks at a glance

"Speed" has no single answer — it depends entirely on where you start and stop the
stopwatch. So we report three:

| # | Name | Stopwatch starts | Stopwatch stops | Answers |
|---|------|------------------|-----------------|---------|
| **1** | Raw forward (resize **NOT** timed) | tensor already on GPU | 640 mask on CPU | *How fast is the architecture?* |
| **2** | Full 640 pipeline (resize **timed**) | RGB image in RAM | 640 mask on CPU | *How fast is inference at 640?* |
| **3** | **Deployment** (recommended) | RGB image in RAM | mask at **native 4022×6024** | *How fast is the real product?* |

**Why benchmark 3 exists.** Benchmarks 1 and 2 both stop at a 640×640 mask. But the
thing we actually ship is a mask overlaid on the **original camera photo**, which is
4022×6024 = **24.2M pixels — 59× more than 640×640**. Upsampling the mask back up is
real work that neither 1 nor 2 counts. Benchmark 3 measures the whole camera-frame-to-
overlayable-mask path, which is the only number a deployment decision should use.

## 1. Mount Google Drive

The trained models and test images live in a zip on Drive
(`bruise_colab_gpu_full.zip`, ~868 MB). We mount Drive to get at it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/bruise_segmentation_gpu'
ZIP_PATH  = f'{DRIVE_DIR}/bruise_colab_gpu_full.zip'

import os
assert os.path.exists(ZIP_PATH), f'{ZIP_PATH} not found — upload the zip to {DRIVE_DIR}/ first.'
print('Found package:', ZIP_PATH, f'({os.path.getsize(ZIP_PATH)/1e6:.0f} MB)')

## 2. Check we actually have a GPU

Timing numbers are meaningless without one. This cell stops the notebook immediately
if the runtime is CPU-only, rather than letting you collect nonsense.

**Runtime → Change runtime type → A100 GPU** if this fails.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > A100 GPU, then re-run.'
DEVICE = torch.device('cuda:0')
print('GPU:', torch.cuda.get_device_name(0))

## 3. Copy the zip to local disk and unzip

We copy off Drive first. Reading images directly from the Drive mount goes over the
network, and that unpredictable delay would leak into the timings we're about to
measure. Local disk keeps the benchmark honest.

In [ ]:
import shutil, time
LOCAL_DIR = '/content/bruise_gpu'

t0 = time.time()
shutil.copy(ZIP_PATH, '/content/pkg.zip')
print(f'Copied in {time.time()-t0:.0f}s')

!rm -rf {LOCAL_DIR}
!mkdir -p {LOCAL_DIR}
t0 = time.time()
!unzip -q /content/pkg.zip -d {LOCAL_DIR}
print(f'Unzipped in {time.time()-t0:.0f}s')

## 4. Install the libraries

Colab already ships torch + CUDA, so we never reinstall torch (that would break the
GPU build). We only add the four libraries our pipeline needs.

In [ ]:
%cd {LOCAL_DIR}
!pip install -q transformers ultralytics albumentations opencv-python-headless

## 5. Imports and configuration

Note the import order: **`cv2` before `ultralytics`**. On some setups importing
ultralytics first makes `cv2.imread(..., IMREAD_GRAYSCALE)` return shape `(H,W,1)`
instead of `(H,W)`, which silently corrupts mask handling. Cheap to avoid.

In [ ]:
import cv2                       # before ultralytics, on purpose
import numpy as np
import pandas as pd
import torch, torch.nn.functional as F
import statistics, sys, json
from torch.utils.data import DataLoader
from ultralytics import YOLO
from ultralytics.data.augment import LetterBox

sys.path.insert(0, LOCAL_DIR)
from pipeline.io_utils import load_yaml
from pipeline.data import BruiseDataset, load_fixed_test
from pipeline.models import load_segformer_model, count_params
from pipeline.metrics import compute_image_row

paths = load_yaml('configs/paths.yaml')
cfg   = load_yaml('configs/common_train.yaml')
IMG_H = IMG_W = 640                      # every model runs and is scored at 640x640
assert (cfg['img_h'], cfg['img_w']) == (IMG_H, IMG_W)

torch.backends.cudnn.benchmark = True    # autotune conv kernels; safe, input shape is fixed
print('Config loaded. Model grid:', IMG_H, 'x', IMG_W)

## 6. Load the test dataset

`BruiseDataset` (from `pipeline/data.py`) is the same loader used during training. For
each of the 185 test images it:

1. reads the photo (native **4022×6024**) and its ground-truth mask,
2. **resizes both to 640×640** — image bilinear, mask nearest-neighbour (nearest keeps
   the mask strictly 0/1; bilinear would invent grey half-pixels),
3. applies **ImageNet normalisation** to the image,
4. returns tensors.

Because prediction and ground truth land on the *same* 640 grid, Dice is a fair
comparison. `batch_size=1` because we want per-image numbers, not batch averages.

**v2 addition (per_model_batch/final style).** Below, we also build a **640 stretch cache once** and **stage all 185 test images on the GPU** as `/255` tensors, so the accuracy pass runs entirely on resident GPU tensors (no per-image disk read/resize). Each model just rescales in the loop — SegFormer to ImageNet, YOLO stays `/255`.

In [ ]:
test_df = load_fixed_test(paths['fixed_test_manifest'])
test_ds = BruiseDataset(test_df, IMG_H, IMG_W, training=False)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=2, pin_memory=True)

x, y, stem, img_path, mask_path = test_ds[0]
print(f'Test images      : {len(test_df)}')
print(f'Image tensor     : {tuple(x.shape)}  (3 channels, 640x640, ImageNet-normalised)')
print(f'GT mask tensor   : {tuple(y.shape)}  (1 channel, 640x640, values 0/1)')
print(f'Native photo size: {cv2.imread(str(img_path)).shape}  <-- what the camera gives us')

# === [per_model_batch/final] 640 CACHE + GPU STAGING =========================================
# Build the 640x640 stretch cache ONCE (bilinear image, nearest mask -- same geometry as
# BruiseDataset), then stage all 185 test images on the GPU as /255 tensors. The accuracy pass
# runs on these resident tensors and each model rescales in-loop (SegFormer -> ImageNet norm,
# YOLO stays /255). This is exactly per_model_batch/final's neutral-dataloader design.
from pathlib import Path
CACHE640 = Path(LOCAL_DIR) / 'cache640'

def build_cache(df, split):
    idir = CACHE640 / split / 'images'; mdir = CACHE640 / split / 'masks'
    idir.mkdir(parents=True, exist_ok=True); mdir.mkdir(parents=True, exist_ok=True)
    for _, r in df.iterrows():
        ip = idir / f'{r.stem}.png'; mp = mdir / f'{r.stem}.png'
        if not ip.exists():
            im = cv2.imread(str(r.image_path), cv2.IMREAD_COLOR)
            cv2.imwrite(str(ip), cv2.resize(im, (IMG_W, IMG_H), interpolation=cv2.INTER_LINEAR))
        if not mp.exists():
            gm = cv2.imread(str(r.mask_path), cv2.IMREAD_GRAYSCALE)
            if gm.ndim == 3: gm = gm[..., 0]
            cv2.imwrite(str(mp), cv2.resize((gm > 0).astype(np.uint8), (IMG_W, IMG_H),
                                            interpolation=cv2.INTER_NEAREST) * 255)

if not (CACHE640 / 'test' / 'images').exists():
    _t = time.time(); build_cache(test_df, 'test'); print(f'640 cache built in {time.time()-_t:.0f}s')
else:
    print('640 cache present')

_imgs, _gts, STEMS = [], [], []
for _, r in test_df.iterrows():
    im = cv2.cvtColor(cv2.imread(str(CACHE640/'test'/'images'/f'{r.stem}.png'), cv2.IMREAD_COLOR),
                      cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0          # /255, model-agnostic
    _imgs.append(torch.from_numpy(im.transpose(2, 0, 1)))
    gm = cv2.imread(str(CACHE640/'test'/'masks'/f'{r.stem}.png'), cv2.IMREAD_GRAYSCALE)
    if gm.ndim == 3: gm = gm[..., 0]
    _gts.append(torch.from_numpy((gm > 0).astype(np.uint8)))
    STEMS.append(str(r.stem))

X_TEST_GPU = torch.stack(_imgs).to(DEVICE)          # [185,3,640,640] /255, resident on GPU
GT_640     = torch.stack(_gts)                       # [185,640,640] uint8, on CPU (numpy metrics)
del _imgs, _gts
assert X_TEST_GPU.shape == (len(test_df), 3, IMG_H, IMG_W), X_TEST_GPU.shape
assert GT_640.shape == (len(test_df), IMG_H, IMG_W), GT_640.shape
print(f'Staged on GPU  : {tuple(X_TEST_GPU.shape)} '
      f'= {X_TEST_GPU.element_size()*X_TEST_GPU.nelement()/1e9:.2f} GB (/255)')
print(f'GT on CPU      : {tuple(GT_640.shape)}  (0/1)')

## 7. Load the 5 trained models

Two different loaders, because the two families were trained in different frameworks:

* **SegFormer** (B2 teacher, B0 direct, B0 distilled) — `load_segformer_model()` reads
  `best_model.pt` and **also returns that model's threshold**. The threshold was chosen
  on the *validation* set during training and is frozen; we never tune it on test.
  Prediction = `sigmoid(logit) >= threshold`.
* **YOLO** (direct, distilled) — loaded through Ultralytics. YOLO has **no threshold**:
  its native postprocessing takes an `argmax` over the 2 class channels, which fixes the
  operating point by construction.

In [ ]:
import contextlib, warnings
from transformers.utils import logging as hf_logging

@contextlib.contextmanager
def quiet_pretrained_load():
    """HF reports the decode_head as newly initialised because we start from an
    ImageNet classification checkpoint; load_segformer_model() then loads our
    fine-tuned weights over it with strict key matching. Errors still raise."""
    prev = hf_logging.get_verbosity()
    hf_logging.set_verbosity_error()
    hf_logging.disable_progress_bar()
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            yield
    finally:
        hf_logging.set_verbosity(prev)
        hf_logging.enable_progress_bar()
MODELS = {}

# --- SegFormer: returns a validation-selected threshold with each model ---
for run, disp, pkey in [
    ('segformer_b2_teacher',   'SegFormer-B2 (teacher)',   'segformer_b2_pretrained'),
    ('segformer_b0_direct',    'SegFormer-B0 (direct)',    'segformer_b0_pretrained'),
    ('segformer_b0_distilled', 'SegFormer-B0 (distilled)', 'segformer_b0_pretrained'),
]:
    with quiet_pretrained_load():
        model, threshold, ckpt = load_segformer_model(run, pkey, paths, DEVICE)
    model = model.to(DEVICE).eval()
    MODELS[run] = {'display': disp, 'family': 'segformer', 'model': model,
                   'threshold': threshold, 'params_m': count_params(model)[0]/1e6}

# --- YOLO: wrapper (for native .predict) + raw module (for timing the forward pass) ---
import copy
for run, disp in [('yolo_sem_direct', 'YOLO26n-sem (direct)'),
                  ('yolo_sem_distilled', 'YOLO26n-sem (distilled)')]:
    wrapper = YOLO(f'{run}/ultralytics_runs/train/weights/best.pt')
    nn_model = copy.deepcopy(wrapper.model).to(DEVICE).eval()
    MODELS[run] = {'display': disp, 'family': 'yolo', 'wrapper': wrapper, 'model': nn_model,
                   'threshold': None,
                   'params_m': sum(p.numel() for p in nn_model.parameters())/1e6}

for k, v in MODELS.items():
    thr = 'argmax (no threshold)' if v['threshold'] is None else f"threshold = {v['threshold']:.2f}"
    print(f"{v['display']:26s} {v['params_m']:6.2f}M params | {thr}")

## 8. Preprocessing and postprocessing — spelled out here

**This is the most important cell to read.** The two model families need *different*
preprocessing, because they were trained differently:

| | SegFormer | YOLO |
|---|---|---|
| Resize | stretch to 640×640 | **stretch to 640×640** (v2: letterbox removed — same grid as the dataloader) |
| Scaling | **ImageNet** mean/std | plain **/255** |
| Mask from output | `sigmoid(logit) >= threshold` | `argmax` over 2 channels |

**Why this matters so much:** we once fed YOLO an ImageNet-normalised tensor (SegFormer's
recipe). It didn't crash — it just quietly scored **0.506 mean Dice instead of 0.735**.
A model only works on the input distribution it was trained on. Getting this wrong is
silent, so we write both paths out explicitly rather than sharing one function.

In [ ]:
"""comment the letter box - modular inference code."""

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
_letterbox = LetterBox(new_shape=(IMG_H, IMG_W))            # kept (unused): v2 uses stretch resize

# ImageNet mean/std as GPU tensors -- used to rescale the neutral /255 staged tensors to
# SegFormer's ImageNet input in the timing + accuracy loops (YOLO stays /255).
_MEAN_T = torch.tensor(IMAGENET_MEAN, device=DEVICE).view(1, 3, 1, 1)
_STD_T  = torch.tensor(IMAGENET_STD,  device=DEVICE).view(1, 3, 1, 1)

def preprocess_segformer(img_rgb):
    """Native RGB photo -> [1,3,640,640] GPU tensor, ImageNet-normalised (stretch resize)."""
    r = cv2.resize(img_rgb, (IMG_W, IMG_H), interpolation=cv2.INTER_LINEAR)
    r = (r.astype(np.float32) / 255.0 - IMAGENET_MEAN) / IMAGENET_STD
    return torch.from_numpy(r).permute(2, 0, 1).unsqueeze(0).to(DEVICE)

def preprocess_yolo(img_rgb):
    """Native RGB photo -> [1,3,640,640] GPU tensor, /255 only (STRETCH resize, no letterbox)."""
    # r = _letterbox(image=img_rgb)                     # v1 letterbox -- commented out per v2
    r = cv2.resize(img_rgb, (IMG_W, IMG_H), interpolation=cv2.INTER_LINEAR)   # stretch to 640
    t = torch.from_numpy(r).permute(2, 0, 1).float().div(255.0).unsqueeze(0)
    return t.to(DEVICE)

def postprocess_segformer(logits, threshold):
    """[1,1,640,640] logits -> 640x640 uint8 mask on CPU."""
    prob = torch.sigmoid(logits[:, 0])
    return (prob >= threshold).to(torch.uint8)[0].cpu().numpy()

def _yolo_bruise_prob(preds):
    """YOLO raw output -> [B,640,640] bruise probability = sigmoid(z1 - z0) (upsampled to 640)."""
    if isinstance(preds, (tuple, list)):
        preds = preds[0]
    preds = preds.float()
    if preds.shape[-2:] != (IMG_H, IMG_W):
        preds = F.interpolate(preds, size=(IMG_H, IMG_W), mode='bilinear', align_corners=False)
    return torch.sigmoid(preds[:, 1] - preds[:, 0])

def postprocess_yolo(preds):
    """argmax mask (native rule == threshold at prob 0.5). Fallback if no swept threshold."""
    return (_yolo_bruise_prob(preds) >= 0.5).to(torch.uint8)[0].cpu().numpy()

def postprocess_yolo_prob(preds, threshold):
    """sigmoid(z1 - z0) >= val-swept threshold -> 640x640 uint8 mask (symmetric with SegFormer)."""
    return (_yolo_bruise_prob(preds) >= threshold).to(torch.uint8)[0].cpu().numpy()

print('Preprocessing / postprocessing defined for both families.')

---
# YOLO threshold — swept on the val set (now symmetric with SegFormer)

SegFormer carries a val-calibrated threshold; YOLO was using bare `argmax` (= threshold at
prob 0.5). Here we give YOLO a threshold the same way: sweep `sigmoid(z1 − z0) ≥ thr` on the
**134 val images** (subject-disjoint from test), pick the best val Dice, and apply it once to
test — fit on val, never on test, so no leakage. Expect YOLO's test Dice/miss to shift a
little (argmax is just the special case thr = 0.5).

In [ ]:
# Load val (subject-disjoint from test), stage /255 on the GPU, sweep a YOLO prob threshold.
val_df = load_fixed_test(f'{LOCAL_DIR}/splits/val_split.csv')
assert not (set(val_df.subject) & set(test_df.subject)), 'val/test SUBJECT leak -- wrong package?'
assert not (set(val_df.stem)    & set(test_df.stem)),    'val/test IMAGE leak -- wrong package?'
build_cache(val_df, 'val')
print(f'Val: {len(val_df)} images, {val_df.subject.nunique()} subjects (0 shared with test)')

_vx, _vg = [], []
for _, r in val_df.iterrows():
    im = cv2.cvtColor(cv2.imread(str(CACHE640/'val'/'images'/f'{r.stem}.png'), cv2.IMREAD_COLOR),
                      cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    _vx.append(torch.from_numpy(im.transpose(2, 0, 1)))
    gm = cv2.imread(str(CACHE640/'val'/'masks'/f'{r.stem}.png'), cv2.IMREAD_GRAYSCALE)
    if gm.ndim == 3: gm = gm[..., 0]
    _vg.append(torch.from_numpy(gm > 0))
VAL_X  = torch.stack(_vx).to(DEVICE)     # [134,3,640,640] /255
VAL_GT = torch.stack(_vg).to(DEVICE)     # [134,640,640] bool
del _vx, _vg

def _val_dice_at(prob, gt, thr):
    """Mean per-image Dice at one threshold (empty pred AND empty GT -> 1.0)."""
    pred = prob >= thr
    inter = (pred & gt).sum((1, 2)).float()
    denom = pred.sum((1, 2)).float() + gt.sum((1, 2)).float()
    return torch.where(denom > 0, 2 * inter / denom, torch.ones_like(denom)).mean().item()

THRS = np.linspace(0.05, 0.95, 91)       # probability grid
for run, m in MODELS.items():
    if m['family'] != 'yolo':
        continue                          # SegFormer already carries its val-calibrated threshold
    with torch.no_grad():
        probs = torch.cat([_yolo_bruise_prob(m['model'](VAL_X[i:i+1])) for i in range(VAL_X.shape[0])])
    dices = [_val_dice_at(probs, VAL_GT, t) for t in THRS]
    best = int(np.argmax(dices))
    m['threshold'] = float(THRS[best])
    print(f"{m['display']:26s} swept val threshold = {m['threshold']:.3f}  "
          f"(val Dice {dices[best]:.4f} vs argmax@0.50 {_val_dice_at(probs, VAL_GT, 0.5):.4f})")

del VAL_X, VAL_GT
torch.cuda.empty_cache()

## 9. The timing harness

Two details that make GPU timing correct, and are easy to get wrong:

* **`torch.cuda.synchronize()`** — the GPU runs *asynchronously*. Without this we'd be
  timing how long it takes to *queue* the work (microseconds), not to *do* it
  (milliseconds). We sync before starting and before stopping the clock.
* **Warmup** — the first few calls pay one-off costs (kernel selection, memory
  allocation). We run 30 throwaway iterations first so we measure steady state.

We report **median** and **p95** alongside the mean. The mean gets dragged around by
occasional slow frames; the median is the typical case and p95 is the "how bad does it
get" number that actually matters for a responsive UI.

In [ ]:
N_WARMUP, N_REPEATS = 30, 3        # 185 images x N_REPEATS timed calls per model

def benchmark_forward(entry):
    """Time 640-tensor-on-GPU -> 640-mask-on-GPU over ALL 185 staged test images, N_REPEATS each.

    Matches benchmark_speed in per_model_batch/final: the tensor is already on the GPU (resize
    and host->GPU copy excluded), the mask NEVER leaves the GPU, and cuda.synchronize() brackets
    every call (CUDA is async -- without it we'd time queueing, not compute). One image at a time
    = real single-frame latency.
    """
    model = entry['model']; fam = entry['family']; n = X_TEST_GPU.shape[0]

    def one(x255):
        if fam == 'segformer':
            z = model((x255 - _MEAN_T) / _STD_T)           # [1,1,640,640] logit
            _ = torch.sigmoid(z[:, 0]) >= entry['threshold']   # GPU mask, no CPU copy
        else:
            p = _yolo_bruise_prob(model(x255))              # [1,640,640] prob on GPU
            thr = 0.5 if entry['threshold'] is None else entry['threshold']
            _ = p >= thr

    with torch.no_grad():
        for i in range(N_WARMUP):
            one(X_TEST_GPU[i % n:i % n + 1])
        torch.cuda.synchronize(DEVICE)
        torch.cuda.reset_peak_memory_stats(DEVICE)
        lat = []
        for _ in range(N_REPEATS):
            for i in range(n):
                x = X_TEST_GPU[i:i + 1]                     # already on GPU, already 640, already /255
                torch.cuda.synchronize(DEVICE)
                t0 = time.perf_counter()
                one(x)
                torch.cuda.synchronize(DEVICE)
                lat.append((time.perf_counter() - t0) * 1000.0)

    lat = np.array(lat)
    return {'mean_ms': lat.mean(), 'median_ms': np.median(lat), 'p95_ms': np.percentile(lat, 95),
            'fps': 1000.0 / lat.mean(), 'n_timed': int(lat.size),
            'peak_gpu_mb': torch.cuda.max_memory_allocated(DEVICE) / 1024**2}

NATIVE_H, NATIVE_W = cv2.imread(str(test_df.iloc[0]['image_path'])).shape[:2]   # for metadata only
print(f'Timing: {N_WARMUP} warmup + {len(test_df)} images x {N_REPEATS} reps per model, '
      f'cuda.synchronize() around each; native res {NATIVE_W}x{NATIVE_H}')

---
# Benchmark — raw forward over **all 185** test images, on the GPU

v2: benchmarks 2 (full-640) and 3 (deployment) are removed. This single benchmark matches
`benchmark_speed` in per_model_batch / final: the 185 `/255` tensors are already staged on
the GPU, and we time only **640 tensor → 640 mask** per image, `cuda.synchronize()` around
each call, the mask never leaving the GPU.

**✅ INCLUDED** forward pass · (bilinear upsample for YOLO) · sigmoid/threshold (or argmax)
**❌ EXCLUDED** disk read · resize to 640 · normalisation · host→GPU copy · device→host copy

median = typical per-image latency; p95 = slow-tail.

In [ ]:
bench = {}
for run, m in MODELS.items():
    bench[run] = b = benchmark_forward(m)
    print(f"{m['display']:26s} {b['median_ms']:7.3f} ms (median) | p95 {b['p95_ms']:6.3f} | "
          f"{b['fps']:7.1f} FPS | {b['n_timed']} calls | peak {b['peak_gpu_mb']:6.1f} MB")

---
# Accuracy evaluation — all 5 models on the 185-image test set at 640

Now we stop timing and measure **quality**. Every model is scored on the same 640×640
grid against the same ground truth, so the numbers are directly comparable.

**v2 — one dataloader, GPU-staged, for both families.** All 185 test images are
pre-staged on the GPU as `/255` tensors (from the 640 cache above). In the loop each model
rescales that neutral `/255` tensor to its own recipe: **SegFormer** applies ImageNet norm
(`(x−MEAN)/STD`), **YOLO** uses `/255` as-is. YOLO's mask is `argmax` over its 2 class
channels (its native rule). This drops v1's native `.predict()` + letterbox path, so all 5
models run on one grid, one preprocessing, no per-image disk read. (Feeding YOLO an
ImageNet-normalised tensor would score ~0.5 — keeping it at `/255` is what matters.)

**Metrics.** Dice/IoU measure overlap. **Complete-miss rate** is the fraction of images
with a bruise where the model predicts *nothing* — reported separately because a blank
mask is a missed injury, which is far worse than a slightly loose outline.

In [ ]:
def to_640_nearest(mask):
    """Any binary mask -> 640x640, nearest-neighbour. Squeeze guards against (H,W,1)."""
    m = np.asarray(mask)
    if m.ndim == 3:
        m = m.squeeze(-1)
    return (cv2.resize(m.astype('uint8'), (IMG_W, IMG_H), interpolation=cv2.INTER_NEAREST) > 0).astype('uint8')

# GPU-staged inference over all 185 test images (per_model_batch/final style): /255 tensors are
# already on the GPU; each model rescales in-loop -- SegFormer to ImageNet (threshold), YOLO stays
# /255 and uses its val-SWEPT threshold. No native .predict(), no letterbox, one grid for all 5.
per_image = {}
for run, m in MODELS.items():
    rows = []
    with torch.no_grad():
        for i in range(X_TEST_GPU.shape[0]):
            x255 = X_TEST_GPU[i:i + 1]                        # /255, already on GPU
            gt   = GT_640[i].numpy()
            if m['family'] == 'segformer':
                logits = m['model']((x255 - _MEAN_T) / _STD_T)   # -> ImageNet norm
                pred = postprocess_segformer(logits, m['threshold'])
            else:
                preds = m['model'](x255)                          # /255 as-is
                pred = postprocess_yolo_prob(preds, m['threshold'])   # val-swept threshold
            rows.append(compute_image_row(pred, gt, STEMS[i]))
            if (i + 1) % 50 == 0:
                print(f"  [{m['display']}] {i+1}/{len(test_df)}")

    df = pd.DataFrame(rows)
    df['complete_miss'] = (df.pred_positive_pixels == 0) & (df.gt_positive_pixels > 0)
    per_image[run] = df
    print(f"{m['display']:26s} median Dice={df.dice.median():.3f}  mean={df.dice.mean():.3f}  "
          f"miss={df.complete_miss.mean()*100:.2f}%")

## Final results — accuracy and speed together

`FPS_deployment` is benchmark 3, the honest one.

Read the miss-rate column next to the Dice column: the model with the best Dice is not
necessarily the one you want in the field.

In [ ]:
rows = []
for run, m in MODELS.items():
    df = per_image[run]
    rows.append({
        'model': m['display'],
        'params_M': round(m['params_m'], 2),
        'threshold': round(float(m['threshold']), 3),   # SegFormer: val-calibrated; YOLO: val-swept
        'median_Dice': round(df.dice.median(), 4),
        'mean_Dice': round(df.dice.mean(), 4),
        'median_IoU': round(df.iou.median(), 4),
        'complete_miss_%': round(df.complete_miss.mean()*100, 2),
        'median_ms': round(bench[run]['median_ms'], 3),
        'p95_ms': round(bench[run]['p95_ms'], 3),
        'FPS': round(bench[run]['fps'], 1),
    })
results = pd.DataFrame(rows)
results

## Save everything to Drive

Per-image CSVs (one row per test image per model), the results table, the 3-benchmark
timing table, and a metadata file recording exactly what each stopwatch included.

In [ ]:
import datetime
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
OUT = f'{LOCAL_DIR}/inference_v2'
os.makedirs(OUT, exist_ok=True)

for run, df in per_image.items():
    df.to_csv(f'{OUT}/per_image_{run}.csv', index=False)
results.to_csv(f'{OUT}/results_accuracy_and_speed.csv', index=False)
pd.DataFrame(bench).T.to_csv(f'{OUT}/benchmark_all185.csv')

json.dump({
  'gpu': torch.cuda.get_device_name(0),
  'n_test_images': len(test_df), 'n_val_images': len(val_df), 'scoring_grid': '640x640',
  'native_resolution': f'{NATIVE_W}x{NATIVE_H}',
  'timing_protocol': (f'{N_WARMUP} warmup + {len(test_df)} images x {N_REPEATS} reps per model, '
                      'GPU tensors pre-staged, cuda.synchronize() around each call, mask stays on GPU'),
  'benchmark': {
      'scope': '640 GPU tensor -> 640 GPU mask, ALL 185 images (== per_model_batch benchmark_speed)',
      'included': ['forward pass', 'bilinear upsample (YOLO)', 'sigmoid/threshold or argmax'],
      'excluded': ['disk read', 'resize to 640', 'normalisation', 'host->GPU copy',
                   'device->host copy (mask never leaves the GPU)']},
  'segformer_preprocessing': 'stretch resize to 640 + ImageNet mean/std; val-calibrated threshold',
  'yolo_preprocessing': ('stretch resize to 640 + /255, GPU-staged from a 640 cache; '
                         'threshold sigmoid(z1-z0) SWEPT on 134 val images (no letterbox, no native .predict())'),
  'thresholds': {run: round(float(m['threshold']), 4) for run, m in MODELS.items()},
  'note': ('v2: all 5 models share one dataloader / one 640 grid; YOLO uses a val-swept threshold '
           '(symmetric with SegFormer) instead of argmax; benchmarks 2 & 3 removed; timing over all '
           '185 images. Original bruise_colab_inference.ipynb (native .predict()+letterbox, 3 '
           'benchmarks, 1-image timing) kept for comparison.'),
}, open(f'{OUT}/metadata.json', 'w'), indent=2)

dest = f'{DRIVE_DIR}/inference_v2_{stamp}'
shutil.copytree(OUT, dest)
print('Saved to:', dest)
!ls -la {dest}